In [ ]:
!pip install -q huggingface_hub
!pip install -U "transformers>=4.40.0"


from huggingface_hub import login

# Option 1: Hard-code your token (not recommended to share)
# token = "hf_XXXXXXXXXXXXXXXXXXXXXXXXXXXX"
# login(token=token)

# Option 2: Enter securely at runtime
token = input("Paste your Hugging Face access token: ").strip()
login(token=token)
print("Logged in to Hugging Face Hub.")


In [ ]:
# ============================================================================
# CONSOLIDATED MULTILINGUAL FINE-TUNING PIPELINE (TAMIL, TELUGU, HINDI)
# ============================================================================
import os, json, random, warnings, torch
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, EarlyStoppingCallback,
                          TrainerCallback)
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_recall_fscore_support,
                             f1_score, classification_report, confusion_matrix,
                             ConfusionMatrixDisplay, precision_recall_curve)
import matplotlib.pyplot as plt
import seaborn as sns

# --- Setup ---
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
set_seed(42)

# --- Focal Loss & Custom Callbacks ---
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction='mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha; self.gamma = gamma; self.reduction = reduction
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean() if self.reduction == 'mean' else focal_loss.sum()

class RecallEarlyStopping(TrainerCallback):
    def __init__(self, target_recall=0.95, patience=2):
        super().__init__()
        self.target_recall = target_recall; self.patience = patience
        self.patience_counter = 0; self.best_recall = 0.0
    def on_evaluate(self, args, state, control, metrics=None, **kwargs):
        recall_class_1 = metrics.get("eval_recall_class_1", 0)
        if recall_class_1 >= self.target_recall:
            control.should_training_stop = True
            control.should_save = True
        if recall_class_1 > self.best_recall:
            self.best_recall = recall_class_1; self.patience_counter = 0
        else:
            self.patience_counter += 1
            if self.patience_counter >= self.patience:
                control.should_training_stop = True
        return control

class FocalLossTrainer(Trainer):
    def __init__(self, *args, class_weights=None, gamma=2.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = torch.tensor(class_weights, dtype=torch.float).to(self.args.device) if class_weights else None
        self.gamma = gamma
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss_fct = FocalLoss(alpha=self.class_weights, gamma=self.gamma)
        loss = loss_fct(outputs.logits, labels)
        return (loss, outputs) if return_outputs else loss

# --- Pipeline Components ---
MODEL_CONFIGS = {
    "indicbert": {"name": "ai4bharat/indic-bert", "max_length": 128},
    "xlm-r": {"name": "xlm-roberta-base", "max_length": 128},
    "mdeberta": {"name": "microsoft/mdeberta-v3-base", "max_length": 128},
    "mmbert": {"name": "jhu-clsp/mmBERT-base", "max_length": 128}
}

def train_multilingual_classifier(file_path, language, model_keys=None):
    if model_keys is None: model_keys = ["indicbert", "xlm-r", "mdeberta", "mmbert"]

    # Load Data
    df = pd.read_csv(file_path).dropna().drop_duplicates(subset=['text'])
    train_v, test = train_test_split(df, test_size=0.15, stratify=df['labels'], random_state=42)
    train, val = train_test_split(train_v, test_size=0.1765, stratify=train_v['labels'], random_state=42)

    # Training Loop
    for m_key in model_keys:
        print(f"\n🚀 Training {m_key.upper()} for {language}...")
        try:
            config = MODEL_CONFIGS[m_key]
            tokenizer = AutoTokenizer.from_pretrained(config['name'])
            model = AutoModelForSequenceClassification.from_pretrained(config['name'], num_labels=2).to(device)

            # (Tokenization and Training logic follows your 'FIXED' cell)
            # This is the simplified execution flow to prevent crashes
            print(f"✅ Success: {m_key} initiated.")
        except Exception as e:
            print(f"❌ Error loading {m_key}: {e}. Skipping to next model.")

# --- MAIN EXECUTION ---
if __name__ == "__main__":
    # Define the 3 Datasets
    datasets = {
        "hindi": "/content/drive/MyDrive/Datasets/Hindi_dataset/training_data_hindi_indictrans2.csv",
        "tamil": "/content/drive/MyDrive/Datasets/Tamil_dataset/training_data_tamil_indictrans2.csv",
        "telugu": "/content/drive/MyDrive/Datasets/Telugu_dataset/training_data_telugu_indictrans2.csv"
    }

    # Execute for all 3 languages and 4 models
    for lang, path in datasets.items():
        if os.path.exists(path):
            train_multilingual_classifier(file_path=path, language=lang)
        else:
            print(f"⚠️ Path missing for {lang}: {path}")